In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["KERAS_BACKEND"] = "torch"
import keras
keras.mixed_precision.set_global_policy("mixed_float16")
keras.config.set_image_data_format("channels_first")
from pathlib import Path
import torch

from PIL import Image
import numpy as np
# Local files/code
from src.agents.U_NET import UNET
from src.data_preprocessing.plant_dataset import PlantDataset
import cv2
import matplotlib.pyplot as plt
import src.config as config
from pathlib import Path
from src.data_preprocessing.segmentation import get_percent_of_Disease
import pandas as pd

In [2]:
best = keras.models.load_model(r"models\visual_models\UNet_Best.keras")

In [3]:
def prep_image(path, size=(384,384)):
    img = np.asarray(Image.open(path).convert('RGB'))
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    x = np.transpose(img.astype(np.float32) / 255.0, (2, 0, 1))[None]
    return img, x

In [ ]:
#loop through GBIF
GBIF_IMAGES = config.Data.GBIF.IMAGES

#make segmentedImages_Data_Dir
Path.mkdir(config.Data.GBIF.SEGMENTED, exist_ok=True)
for folder in GBIF_IMAGES.iterdir():
    Mask_Folder = config.Data.GBIF.SEGMENTED / folder.name
    Path.mkdir(Mask_Folder, exist_ok=True)
    for item in folder.iterdir():
        mask_path = Mask_Folder / str(item.stem + '.png')
        if Path.exists(mask_path):
            continue
        #Load the image
        img, x = prep_image(item)
        mask = best.predict(x, verbose=0)
        mask = (mask > 0.7).astype(np.uint8) * 255
        mask = mask.squeeze()
        
        if not cv2.imwrite(mask_path, mask):
            print('failed to save mask', mask_path)
    


KeyboardInterrupt: 

In [ ]:
def clean_path(p: Path) -> str:
    parts = p.resolve().parts
    i = parts.index('data')
    return Path(*parts[i:]).as_posix()

In [ ]:
MASK_DIR = config.Data.GBIF.SEGMENTED 
gbif_df = []
for folder in GBIF_IMAGES.iterdir():
    Mask_Folder = config.Data.GBIF.SEGMENTED / folder.name
    for item in folder.iterdir():
        mask_path = Mask_Folder / str(item.stem + '.png')
        if not mask_path.exists():
            plant_dict = {'symbol' : folder.name,
                                  'img_id' : item.stem,
                                  'img_path': item,
                                  'mask_path': 'NO MASK',
                                  'disease' : 'NO MASK',
                                  'severity' : 'NO MASK'}
            continue

        #Load the image
        dis, sev = get_percent_of_Disease(item, mask_path)
        plant_dict = {'symbol' : folder.name,
                      'img_id' : item.stem,
                      'img_path': clean_path(item),
                      'mask_path': clean_path(mask_path),
                      'disease' : dis,
                      'severity' : sev}
        gbif_df.append(plant_dict)
final_df = pd.DataFrame(gbif_df)
final_df.to_csv('GBIF_UPDATED_SEVERITY.csv', index=False)